<a href="https://colab.research.google.com/github/sebastiangome/latam-agente/blob/main/LATAM_RAG_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente de Cuenta LATAM Airlines — Pipeline RAG (ISY0101, EP1)

Notebook del encargo **Diseño de Solución con LLM y RAG**, construido con el stack del curso (RA1/IL1.3 y RA1/IL1.1):

- **LangChain** como framework de orquestación.
- **Groq** (`ChatGroq`) para el LLM de chat (`llama-3.3-70b-versatile`).
- **Gemini** (`GoogleGenerativeAIEmbeddings`) para los embeddings (`text-embedding-004`), con fallback local (`HuggingFaceEmbeddings`).
- **FAISS** (vía `langchain_community.vectorstores`) como base de datos vectorial con metadata trazable.

**Caso:** LATAM Airlines — chatbot que responde consultas de pasajeros sobre cambios de vuelo, equipaje y Millas LATAM Pass, combinando:

- **Fuente interna (simulada):** perfil de reserva del pasajero (tarifa, cabina, ruta, categoría LATAM Pass, millas).
- **Fuente externa (real):** resúmenes normativos de páginas oficiales del Centro de Ayuda de LATAM (citas y trazabilidad por URL).

> **Credenciales necesarias:**
> - `LLM_API_KEY` (Groq, gratis en [console.groq.com/keys](https://console.groq.com/keys))
> - `GOOGLE_API_KEY` (Gemini, gratis en [aistudio.google.com/apikey](https://aistudio.google.com/apikey))
> Cárgalas como **Secrets de Colab** (icono 🔑) con esos nombres exactos, activando el interruptor **"Notebook access"**. En entorno local, configúralas en tu archivo `.env`.

## MÓDULO 1 — Setup y Configuración de Modelos

**Alcance técnico:**
- Configuración e instalación de dependencias (Google Colab / Entorno local).
- Carga segura de credenciales (`LLM_API_KEY`, `GOOGLE_API_KEY`) desde Secrets o `.env`.
- Inicialización del LLM oficial (`ChatGroq` para Groq) y Embeddings (`GoogleGenerativeAIEmbeddings` / Fallback HuggingFace).

## 1. Instalación de dependencias

In [1]:
# Instalacion de dependencias (para Google Colab)
!pip install -q faiss-cpu langchain langchain-community langchain-core langchain-google-genai langchain-openai openai python-dotenv langchain-huggingface sentence-transformers langchain-groq groq



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Users\sebas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Carga segura de credenciales y librerías

Intenta leer los Secrets de Google Colab (`userdata`); si se ejecuta en entorno local, lee las variables desde un archivo `.env`.

In [2]:
import os
import random
import re
import subprocess
import sys
import numpy as np
import pandas as pd

# 1. Carga de credenciales desde Colab Secrets o .env local
EN_COLAB = False
try:
    from google.colab import userdata
    EN_COLAB = True
    print("[INFO] Entorno detectado: Google Colab. Buscando Secrets...")

    for _k in ("LLM_API_KEY", "GROQ_API_KEY", "GOOGLE_API_KEY", "LLM_MODEL"):
        try:
            _valor = userdata.get(_k)
            os.environ[_k] = _valor
            print(f"[OK] Secret '{_k}' cargado correctamente.")
        except userdata.SecretNotFoundError:
            print(f"[INFO] Secret '{_k}' no existe en este notebook (puede ser normal si usas otro nombre).")
        except userdata.NotebookAccessError:
            print(f"[ERROR] Secret '{_k}' existe pero el NOTEBOOK NO TIENE ACCESO. "
                  f"Ve al panel de Secrets (icono llave a la izquierda) y activa el interruptor "
                  f"'Notebook access' junto a '{_k}'.")
        except Exception as _e:
            print(f"[ERROR] No se pudo leer el secret '{_k}': {_e}")

    os.environ.setdefault("LLM_MODEL", "openai/gpt-oss-120b")
except ImportError:
    print("[INFO] Entorno detectado: local (no Colab). Cargando variables desde archivo .env...")
    from dotenv import load_dotenv
    _cargado = load_dotenv()
    if not _cargado:
        print("[ERROR] No se encontro un archivo .env en el directorio actual, o esta vacio. "
              "Copia '.env.example' a '.env' y completa tus claves antes de continuar.")

# Sincronizar variables de entorno para Groq
if os.getenv("LLM_API_KEY") and not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = os.environ["LLM_API_KEY"]
if os.getenv("GROQ_API_KEY") and not os.getenv("LLM_API_KEY"):
    os.environ["LLM_API_KEY"] = os.environ["GROQ_API_KEY"]

# Diagnostico final, ANTES de seguir con el resto del notebook
print("\n" + "=" * 70)
print("DIAGNOSTICO DE CREDENCIALES")
print(f"  LLM_API_KEY (Groq):    {'[OK] detectada' if os.getenv('LLM_API_KEY') else '[FALTA] no detectada'}")
print(f"  GOOGLE_API_KEY (Gemini): {'[OK] detectada' if os.getenv('GOOGLE_API_KEY') else '[FALTA] no detectada'}")
if EN_COLAB and not (os.getenv("LLM_API_KEY") and os.getenv("GOOGLE_API_KEY")):
    print("  -> Revisa: nombres EXACTOS de los Secrets ('LLM_API_KEY' y 'GOOGLE_API_KEY'),")
    print("     el interruptor 'Notebook access' activado, y reinicia el entorno de ejecucion")
    print("     (Entorno de ejecucion > Reiniciar sesion) despues de agregar/editar un Secret.")
print("=" * 70 + "\n")

# 2. Importación con auto-instalación de respaldo (por si no se ejecutó la Celda 1)
try:
    from langchain_groq import ChatGroq
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    from langchain_community.vectorstores import FAISS
    from langchain_core.messages import HumanMessage, SystemMessage
except ImportError:
    print("[INFO] Instalando librerías necesarias en segundo plano...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "faiss-cpu", "langchain", "langchain-community", "langchain-core",
        "langchain-google-genai", "langchain-groq", "groq", "python-dotenv",
        "langchain-huggingface", "sentence-transformers"
    ])
    from langchain_groq import ChatGroq
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    from langchain_community.vectorstores import FAISS
    from langchain_core.messages import HumanMessage, SystemMessage

random.seed(42)
np.random.seed(42)


[INFO] Entorno detectado: local (no Colab). Cargando variables desde archivo .env...

DIAGNOSTICO DE CREDENCIALES
  LLM_API_KEY (Groq):    [OK] detectada
  GOOGLE_API_KEY (Gemini): [OK] detectada



C:\Users\sebas\AppData\Local\Temp\ipykernel_18792\2057449722.py:60: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 3. Configuración de los modelos (Chat LLM + Embeddings)

In [3]:
# === 1. Validacion de credenciales y parametros ===
llm_api_key = (os.getenv("LLM_API_KEY") or os.getenv("GROQ_API_KEY") or "").strip()
llm_model_env = (os.getenv("LLM_MODEL") or "").strip()
google_key = (os.getenv("GOOGLE_API_KEY") or "").strip()

if not llm_api_key:
    print("\n" + "=" * 70)
    print("AVISO: Falta la clave LLM_API_KEY (Groq).")
    print("Crea tu clave gratuita en: https://console.groq.com/keys")
    print("En Colab: Panel lateral > Secrets > Anadir Secret")
    print("  Nombre: LLM_API_KEY  |  Valor: gsk_...")
    print("  Activa el interruptor 'Notebook access'.")
    print("=" * 70 + "\n")
else:
    print(f"[OK] LLM_API_KEY detectada (inicio: {llm_api_key[:8]}...)")

# === 2. Inicializacion del LLM con seleccion automatica de modelo Groq ===
# Lista de modelos en orden de preferencia (del mejor al mas basico disponible)
# NOTA: Groq deprecó/decomisionó llama-3.3-70b-versatile, llama-3.1-8b-instant,
# llama3-70b-8192, llama3-8b-8192, mixtral-8x7b-32768 y gemma2-9b-it (ver
# https://console.groq.com/docs/deprecations). Los modelos de produccion vigentes
# a la fecha son estos (verifica igual en https://console.groq.com/docs/models,
# el catalogo de Groq cambia con frecuencia):
GROQ_MODELOS_CANDIDATOS = [
    llm_model_env,
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "qwen/qwen3.6-27b",
    "llama-3.3-70b-versatile",  # fallback por si tu cuenta aun lo tiene activo
]
# Quitar vacios/duplicados manteniendo orden
GROQ_MODELOS_CANDIDATOS = list(dict.fromkeys(m for m in GROQ_MODELOS_CANDIDATOS if m))

llm = None
llm_model = None
for _modelo in GROQ_MODELOS_CANDIDATOS:
    try:
        _candidato = ChatGroq(
            groq_api_key=llm_api_key,
            model_name=_modelo,
            temperature=0.2,
            max_tokens=500,
        )
        # Llamada de prueba rapida para validar acceso al modelo
        _candidato.invoke([HumanMessage(content="ok")])
        llm = _candidato
        llm_model = _modelo
        print(f"[OK] LLM configurado con ChatGroq ({llm_model}).")
        break
    except Exception as _e:
        print(f"[INFO] Modelo {_modelo!r} no disponible: {_e}")

if llm is None:
    raise RuntimeError(
        "No se pudo inicializar ningun modelo Groq. "
        "Verifica tu clave LLM_API_KEY en https://console.groq.com/keys"
    )

# === 3. Inicializacion de Embeddings (Gemini con fallback HuggingFace) ===
embeddings = None

if google_key:
    modelos_a_probar = [
        "models/text-embedding-004",
        "models/gemini-embedding-001",
        "text-embedding-004",
    ]
    for modelo_nombre in modelos_a_probar:
        try:
            emb_candidato = GoogleGenerativeAIEmbeddings(
                model=modelo_nombre,
                google_api_key=google_key,
            )
            _ = emb_candidato.embed_query("test conexion")
            embeddings = emb_candidato
            print(f"[OK] Embeddings Gemini configurados con '{modelo_nombre}'.")
            break
        except Exception:
            pass

if embeddings is None:
    print("[INFO] Inicializando Embeddings locales multilingues (HuggingFace) como fallback...")
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
        print("[OK] Embeddings locales HuggingFace (paraphrase-multilingual-MiniLM-L12-v2) listos.")
    except Exception:
        from langchain_community.embeddings import HuggingFaceEmbeddings
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
        print("[OK] Embeddings locales HuggingFace (paraphrase-multilingual-MiniLM-L12-v2) listos.")

print(f"[OK] Setup completo: LLM ({llm_model}) y Embeddings ({embeddings.__class__.__name__}) listos.\n")


[OK] LLM_API_KEY detectada (inicio: gsk_czeA...)


[OK] LLM configurado con ChatGroq (openai/gpt-oss-120b).


[OK] Embeddings Gemini configurados con 'models/gemini-embedding-001'.
[OK] Setup completo: LLM (openai/gpt-oss-120b) y Embeddings (GoogleGenerativeAIEmbeddings) listos.



## MÓDULO 2 — Datos Internos Simulados y Políticas Oficiales Externas

**Alcance técnico:**
- Modelado y generación del dataset sintético de reservas de pasajeros (`reservas_latam.csv`).
- Construcción de perfiles narrativos individuales aptos para búsqueda RAG.
- Curaduría de chunks oficiales de políticas públicas del Centro de Ayuda de LATAM con metadata y URLs.

## 4. Generación de datos sintéticos de reservas

Se simulan perfiles de reserva de pasajeros con los campos relevantes para responder consultas de cambios, equipaje y millas: código de reserva, tarifa, cabina, ruta y categoría LATAM Pass.

In [4]:
TARIFAS = ["Light", "Plus", "Top", "Full"]
CABINAS = ["Economy", "Premium Economy", "Premium Business"]
CATEGORIAS_PASS = ["Red", "Silver", "Gold", "Platinum", "Black", "Black Signature"]
RUTAS = ["Santiago-Lima", "Santiago-Bogota", "Santiago-Sao Paulo", "Santiago-Miami", "Santiago-Buenos Aires"]

def generar_codigo_reserva():
    letras = "".join(random.choices("ABCDEFGHJKLMNPQRSTUVWXYZ", k=3))
    numeros = "".join(random.choices("0123456789", k=3))
    return letras + numeros

def generar_reservas(n=200):
    filas = []
    codigos_usados = set()
    for _ in range(n):
        codigo = generar_codigo_reserva()
        while codigo in codigos_usados:
            codigo = generar_codigo_reserva()
        codigos_usados.add(codigo)

        tarifa = random.choice(TARIFAS)
        cabina = random.choice(CABINAS)
        ruta = random.choice(RUTAS)
        categoria = np.random.choice(CATEGORIAS_PASS, p=[0.45, 0.25, 0.15, 0.08, 0.05, 0.02])
        millas = int(np.random.exponential(15000))

        filas.append({
            "codigo_reserva": codigo,
            "tarifa": tarifa,
            "cabina": cabina,
            "ruta": ruta,
            "categoria_pass": categoria,
            "millas_acumuladas": millas,
        })
    return pd.DataFrame(filas)

df_reservas = generar_reservas(200)
os.makedirs("data", exist_ok=True)
df_reservas.to_csv("data/reservas_latam.csv", index=False)
df_reservas.to_csv("reservas_latam.csv", index=False)
print(f"[OK] {len(df_reservas)} reservas generadas y guardadas en data/reservas_latam.csv")
df_reservas.head()


[OK] 200 reservas generadas y guardadas en data/reservas_latam.csv


,codigo_reserva,tarifa,cabina,ruta,categoria_pass,millas_acumuladas
0,RAG276,Light,Premium Business,Santiago-Miami,Red,45151
1,ACF657,Full,Economy,Santiago-Miami,Gold,13694
2,QVA863,Plus,Economy,Santiago-Sao Paulo,Red,2543
3,CKJ320,Full,Premium Business,Santiago-Lima,Red,30168
4,ZKP868,Plus,Premium Business,Santiago-Lima,Silver,18468


## 5. Construcción de perfiles narrativos de reserva (chunks internos)

In [5]:
def construir_perfil_reserva(row):
    return (
        f"Reserva {row['codigo_reserva']}: tarifa {row['tarifa']}, cabina {row['cabina']}, "
        f"ruta {row['ruta']}, categoria LATAM Pass {row['categoria_pass']}, "
        f"{row['millas_acumuladas']} millas acumuladas."
    )

df_reservas["perfil_narrativo"] = df_reservas.apply(construir_perfil_reserva, axis=1)
chunks_reservas = df_reservas["perfil_narrativo"].tolist()
print(chunks_reservas[0])


Reserva RAG276: tarifa Light, cabina Premium Business, ruta Santiago-Miami, categoria LATAM Pass Red, 45151 millas acumuladas.


## 6. Fuente externa: resúmenes propios de políticas oficiales de LATAM (chunks externos)

Cada chunk resume el contenido de una página oficial del Centro de Ayuda de LATAM, asociando la URL de origen como metadata para garantizar trazabilidad y referencias precisas.

In [6]:
documentos_externos = [
    {
        "id": "ext_cambios_01",
        "tema": "cambios",
        "fuente_url": "https://www.latamairlines.com/cl/es/centro-ayuda/preguntas/cambios/pasajes/cambiar-vuelo-fecha-pasaje",
        "texto": (
            "Un pasajero puede cambiar la fecha o el vuelo de su pasaje siempre que las condiciones "
            "de su tarifa lo permitan. El cambio debe solicitarse antes de la salida del vuelo original; "
            "en algunos casos tambien es posible cambiarlo despues de iniciado el viaje si la tarifa lo "
            "autoriza. El proceso se realiza desde la seccion 'Mis Viajes', ingresando el codigo de "
            "reserva y el apellido del pasajero."
        ),
    },
    {
        "id": "ext_cambios_02",
        "tema": "cambios",
        "fuente_url": "https://www.latamairlines.com/cl/es/centro-ayuda/preguntas/problemas-vuelo/cambio-itinerario/vuelo-adelantado",
        "texto": (
            "Si LATAM adelanta un vuelo 16 minutos o mas y el pasajero no esta conforme con el nuevo "
            "horario, puede cambiar la hora o fecha del vuelo sin costo, o solicitar la devolucion del "
            "pasaje, siempre que mantenga el mismo destino y cabina del vuelo original. El plazo para "
            "solicitar este cambio es de hasta 12 meses desde la fecha del vuelo original comprado."
        ),
    },
    {
        "id": "ext_equipaje_01",
        "tema": "equipaje",
        "fuente_url": "https://www.latamairlines.com/us/es/centro-ayuda/preguntas/equipaje",
        "texto": (
            "La franquicia de equipaje facturado depende de la tarifa, cabina y ruta del pasaje. En "
            "general, la cabina Economy permite una cantidad de piezas menor que Premium Economy o "
            "Premium Business. Ademas del equipaje facturado, se permite equipaje de mano y un articulo "
            "personal, con limites de peso y tamano que tambien varian segun la tarifa contratada. Es "
            "importante verificar la franquicia especifica en el sitio oficial antes de viajar, ya que "
            "estas politicas pueden cambiar (por ejemplo, LATAM ha reducido en el pasado la cantidad de "
            "piezas gratuitas en algunos vuelos domesticos)."
        ),
    },
    {
        "id": "ext_millas_01",
        "tema": "millas",
        "fuente_url": "https://www.latamairlines.com/es/es/centro-ayuda/preguntas/latam-pass/millas/como-acumular",
        "texto": (
            "Los pasajeros acumulan Millas LATAM Pass al volar con LATAM o con aerolineas asociadas, "
            "asi como en comercios asociados o mediante tarjetas de credito con convenio. Los pasajes "
            "que fueron pagados totalmente con millas no generan acumulacion de nuevas millas ni de "
            "Puntos Calificables."
        ),
    },
    {
        "id": "ext_millas_02",
        "tema": "millas",
        "fuente_url": "https://latampass.latam.com/es_cl/reglamento-2025/acumulacion-y-canje",
        "texto": (
            "El canje de equipaje adicional con Millas LATAM Pass solo puede realizarse dentro del "
            "mismo flujo de canje del pasaje; no es posible canjear equipaje antes o despues de haber "
            "canjeado el pasaje asociado. La cantidad de millas necesarias para un canje depende del "
            "destino y la cabina seleccionada."
        ),
    },
]

chunks_politicas = [d["texto"] for d in documentos_externos]
print(f"{len(chunks_politicas)} chunks de politicas cargados")


5 chunks de politicas cargados


## MÓDULO 3 — Motor RAG: Vectorización y Recuperación Dual

**Alcance técnico:**
- Construcción de índices vectoriales FAISS con metadata enriquecida (`Document`).
- Arquitectura de recuperación dual: búsqueda determinística/exacta para datos de reserva (privacidad) y búsqueda semántica para políticas oficiales.
- Trazabilidad de fuentes hacia URLs del Centro de Ayuda.

## 7. Bases de datos vectoriales (FAISS)

Se emplean dos índices separados:
1. **Índice de Reservas:** Indexación de muestra de reservas internas.
2. **Índice de Políticas con Metadata (`Document`):** Indexación de políticas con metadata (`fuente_url`, `tema`, `id`) para permitir trazabilidad exacta.

In [7]:
import time
from langchain_core.documents import Document


def construir_documentos_politicas(documentos_externos):
    """Convierte cada entrada de `documentos_externos` (dict con texto + fuente_url + tema)
    en un `Document` de LangChain, dejando el texto como contenido indexable y el resto
    como metadata recuperable junto con el resultado de la búsqueda."""
    return [
        Document(
            page_content=doc["texto"],
            metadata={
                "id": doc["id"],
                "tema": doc["tema"],
                "fuente_url": doc["fuente_url"],
            },
        )
        for doc in documentos_externos
    ]


def construir_index_faiss(nombre, embedding_model, *, textos=None, documentos=None, max_retries=3):
    """Crea un índice FAISS a partir de texto plano (`textos=`) o de `Document` ya con
    metadata (`documentos=`). Maneja reintentos con pausa si se alcanza la cuota por
    minuto de Gemini."""
    if (textos is None) == (documentos is None):
        raise ValueError("Debes entregar exactamente uno de: 'textos' o 'documentos'.")
    for intento in range(max_retries):
        try:
            if documentos is not None:
                index = FAISS.from_documents(documents=documentos, embedding=embedding_model)
            else:
                index = FAISS.from_texts(texts=textos, embedding=embedding_model)
            print(f"[OK] Indice FAISS '{nombre}' creado con {index.index.ntotal} vectores.")
            return index
        except Exception as e:
            err_str = str(e)
            if ("429" in err_str or "RESOURCE_EXHAUSTED" in err_str) and intento < max_retries - 1:
                print(f"[AVISO] Cuota por minuto alcanzada al crear '{nombre}'. Esperando 15s para reintentar (intento {intento+1}/{max_retries})...")
                time.sleep(15)
                continue
            raise RuntimeError(
                f"No se pudo crear el indice FAISS '{nombre}'. Detalle original: {e}"
            ) from e


# Muestra representativa de reservas para el vector store interno (25 reservas)
reservas_db = construir_index_faiss("reservas", embeddings, textos=chunks_reservas[:25])

# Chunks de políticas oficiales con metadata
documentos_politicas = construir_documentos_politicas(documentos_externos)
politicas_db = construir_index_faiss("politicas", embeddings, documentos=documentos_politicas)


[OK] Indice FAISS 'reservas' creado con 25 vectores.


[OK] Indice FAISS 'politicas' creado con 5 vectores.


## 8. Retrievers y detección de código de reserva

Dos mecanismos de recuperación:
- **Reservas → Coincidencia EXACTA:** Búsqueda determinística por código PNR (reserva) en `df_reservas` (garantiza privacidad y exactitud).
- **Políticas → Búsqueda SEMÁNTICA:** Vía retriever FAISS (`as_retriever(search_kwargs={"k": 2})`), recuperando texto y URL de fuente oficial.

In [8]:
def detectar_codigo_reserva(query: str):
    """Busca un patrón de código de reserva (3 letras + 3 números, ej. ABC123) en la consulta."""
    match = re.search(r"\b[A-Z]{3}\d{3}\b", query.upper())
    return match.group(0) if match else None


def recuperar_contexto_reserva(query: str):
    """Si la consulta trae un código de reserva válido y existente en df_reservas, devuelve
    su perfil narrativo exacto (búsqueda determinística). Si no hay código o no existe,
    devuelve listas/valores vacíos para que el agente solicite el dato al pasajero."""
    codigo = detectar_codigo_reserva(query)
    if codigo and codigo in df_reservas["codigo_reserva"].values:
        fila = df_reservas[df_reservas["codigo_reserva"] == codigo].iloc[0]
        return [fila["perfil_narrativo"]], codigo
    return [], None


politicas_retriever = politicas_db.as_retriever(search_kwargs={"k": 2})


def recuperar_contexto_politicas(query: str):
    """Recupera los chunks de políticas más relevantes para la consulta.
    
    Devuelve una lista de dicts {"texto", "fuente_url", "tema"} para que el agente
    pueda citar la fuente exacta en su respuesta.
    """
    docs = politicas_retriever.invoke(query)
    return [
        {
            "texto": d.page_content,
            "fuente_url": d.metadata.get("fuente_url", "fuente no disponible"),
            "tema": d.metadata.get("tema", "sin tema"),
        }
        for d in docs
    ]


_codigo_ejemplo = df_reservas.iloc[0]["codigo_reserva"]

print("[Test] Prueba 1 — recuperación EXACTA por código de reserva:")
_ctx_reserva, _codigo_detectado = recuperar_contexto_reserva(f"tengo la reserva {_codigo_ejemplo}")
print(f"  código detectado: {_codigo_detectado}")
print(f"  contexto recuperado: {_ctx_reserva}")

print("\n[Test] Prueba 2 — recuperación SEMÁNTICA de políticas (con fuente citada):")
for _chunk in recuperar_contexto_politicas("¿cuánto equipaje puedo llevar?"):
    print(f"  - [{_chunk['tema']}] {_chunk['texto'][:90]}...")
    print(f"    Fuente: {_chunk['fuente_url']}")

print("\n[Test] Prueba 3 — consulta sin código de reserva (debe devolver contexto vacío):")
_ctx_vacio, _codigo_vacio = recuperar_contexto_reserva("¿puedo cambiar mi vuelo de mañana?")
assert _ctx_vacio == [] and _codigo_vacio is None, "Se esperaba contexto vacío sin código de reserva"
print("  OK: sin código de reserva -> contexto vacío (el agente deberá pedir el dato).")


[Test] Prueba 1 — recuperación EXACTA por código de reserva:
  código detectado: RAG276
  contexto recuperado: ['Reserva RAG276: tarifa Light, cabina Premium Business, ruta Santiago-Miami, categoria LATAM Pass Red, 45151 millas acumuladas.']

[Test] Prueba 2 — recuperación SEMÁNTICA de políticas (con fuente citada):


  - [equipaje] La franquicia de equipaje facturado depende de la tarifa, cabina y ruta del pasaje. En gen...
    Fuente: https://www.latamairlines.com/us/es/centro-ayuda/preguntas/equipaje
  - [millas] El canje de equipaje adicional con Millas LATAM Pass solo puede realizarse dentro del mism...
    Fuente: https://latampass.latam.com/es_cl/reglamento-2025/acumulacion-y-canje

[Test] Prueba 3 — consulta sin código de reserva (debe devolver contexto vacío):
  OK: sin código de reserva -> contexto vacío (el agente deberá pedir el dato).


## MÓDULO 4 — Agente Conversacional, Prompt Engineering y Evaluación
**Responsable:** Rafael Pacheco (Integrante 2)

**Alcance técnico:**
- Formulación del `SYSTEM_PROMPT` con restricciones de no alucinación y verificación de vigencia.
- Ensamble del prompt aumentado multi-fuente (reserva + políticas + consulta).
- Función orquestadora `consultar_agente()` y ejecución de casos de prueba demostrativos.

## 9. Prompt aumentado y llamada al LLM

Se ensambla el prompt combinando ambas fuentes (reserva + políticas oficiales) junto a las instrucciones del sistema.

In [9]:
SYSTEM_PROMPT = """Eres un asistente de atencion al pasajero para LATAM Airlines.
Tu funcion es responder consultas sobre cambios de vuelo, equipaje y Millas LATAM Pass.

Reglas:
- Usa SOLO la informacion entregada en el CONTEXTO (reserva del pasajero, si existe, y politicas
  oficiales). No inventes datos ni reglas que no esten en el contexto.
- Si la consulta depende de un dato que no tienes (ej. tarifa contratada) y no hay una reserva
  asociada en el contexto, pide ese dato al pasajero en vez de asumirlo.
- Las politicas de LATAM pueden cambiar: siempre recomienda verificar la vigencia en el sitio
  oficial (latamairlines.com) antes de tomar una decision final.
- No ejecutas transacciones reales (cambios, canjes); solo informas y orientas.
- Responde en español, en un tono claro y cercano para un pasajero."""

def construir_prompt(query, contexto_reserva, contexto_politicas):
    partes = []
    if contexto_reserva:
        partes.append("=== CONTEXTO: RESERVA DEL PASAJERO ===\n" + "\n".join(contexto_reserva))
    else:
        partes.append("=== CONTEXTO: RESERVA DEL PASAJERO ===\nNo se identifico una reserva asociada a esta consulta.")

    politicas_texto = "\n---\n".join(
        f"{p['texto']}\n(Fuente: {p['fuente_url']})" for p in contexto_politicas
    )
    partes.append("=== CONTEXTO: POLITICAS OFICIALES LATAM ===\n" + politicas_texto)
    partes.append(f"=== CONSULTA DEL PASAJERO ===\n{query}")
    return "\n\n".join(partes)

def consultar_agente(query):
    contexto_reserva, codigo = recuperar_contexto_reserva(query)
    contexto_politicas = recuperar_contexto_politicas(query)
    prompt_final = construir_prompt(query, contexto_reserva, contexto_politicas)

    respuesta = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=prompt_final),
    ])
    return respuesta.content, contexto_reserva, contexto_politicas


## 10. Demo — consultas de ejemplo

Evidencia para el informe técnico y demostración en vivo.

In [10]:
codigo_ejemplo = df_reservas.iloc[0]["codigo_reserva"]
print("Reserva de ejemplo:", codigo_ejemplo)

pregunta_1 = f"Tengo la reserva {codigo_ejemplo}, ¿cuánto equipaje puedo llevar?"
respuesta_1, ctx_r1, ctx_p1 = consultar_agente(pregunta_1)
print("PREGUNTA:", pregunta_1)
print("\nRESPUESTA DEL AGENTE:\n", respuesta_1)


Reserva de ejemplo: RAG276


PREGUNTA: Tengo la reserva RAG276, ¿cuánto equipaje puedo llevar?

RESPUESTA DEL AGENTE:
 ¡Hola! Con gusto te confirmo la franquicia de equipaje para tu reserva **RAG276**.

- **Tarifa:** Light  
- **Cabina:** Premium Business  
- **Ruta:** Santiago → Miami  

En la política de LAT America, la cantidad de piezas de equipaje facturado depende de la tarifa, la cabina y la ruta. Al viajar en **Premium Business** (aunque sea con tarifa Light) la franquicia de equipaje facturado es mayor que la de la clase Economy y suele incluir **dos piezas** de equipaje facturado, cada una con un peso máximo que normalmente ronda los 32 kg.  

Además, tienes derecho a:

- **Equipaje de mano** (una pieza, con límite de peso y dimensiones según la tarifa y la cabina).  
- **Artículo personal** (por ejemplo, una bolsa de laptop o un bolso pequeño).

**Importante:**  
- Las condiciones exactas (peso, dimensiones y número de piezas) pueden variar y están sujetas a cambios. Te recomiendo que verifiques la fran

In [11]:
pregunta_2 = "¿Cómo acumulo Millas LATAM Pass si no he volado hace tiempo?"
respuesta_2, ctx_r2, ctx_p2 = consultar_agente(pregunta_2)
print("PREGUNTA:", pregunta_2)
print("\nRESPUESTA DEL AGENTE:\n", respuesta_2)


PREGUNTA: ¿Cómo acumulo Millas LATAM Pass si no he volado hace tiempo?

RESPUESTA DEL AGENTE:
 ¡Hola! 😊

Acumular Millas LATAM Pass no depende únicamente de haber volado recientemente. Puedes seguir sumando millas de las siguientes maneras:

| **Actividad** | **Cómo funciona** |
|---------------|-------------------|
| **Volar con LATAM** | Cada vuelo que realices con LATAM o con sus aerolíneas asociadas genera millas según la tarifa y la distancia del recorrido. |
| **Volar con aerolíneas asociadas** | Los vuelos operados por aerolíneas socias del programa también acreditan millas, siempre que el número de ticket o el código de reserva se registre en tu cuenta LATAM Pass. |
| **Compras en comercios asociados** | Tiendas, hoteles, agencias de alquiler de autos y otros comercios afiliados al programa permiten acumular millas al presentar tu número de LATAM Pass o al usar la tarjeta de socio en el momento de la compra. |
| **Tarjetas de crédito con convenio** | Si tienes una tarjeta de cr

In [12]:
pregunta_3 = "¿Puedo cambiar mi vuelo de mañana?"
respuesta_3, ctx_r3, ctx_p3 = consultar_agente(pregunta_3)
print("PREGUNTA:", pregunta_3)
print("\nRESPUESTA DEL AGENTE (sin codigo de reserva, deberia pedir mas datos):\n", respuesta_3)


PREGUNTA: ¿Puedo cambiar mi vuelo de mañana?

RESPUESTA DEL AGENTE (sin codigo de reserva, deberia pedir mas datos):
 ¡Hola! Claro que sí, es posible cambiar la fecha o el vuelo siempre que la tarifa que contrataste lo permita y siempre que la solicitud se haga antes de la salida del vuelo original.

Para poder indicarte con exactitud si tu pasaje es elegible para el cambio, cuál sería el costo (si lo hubiera) y los pasos específicos que debes seguir, necesito que me proporciones los siguientes datos de tu reserva:

1. **Código de reserva (PNR)**  
2. **Apellido del pasajero**  

Con esa información podré revisar las condiciones de tu tarifa y orientarte sobre cómo realizar el cambio a través de la sección **“Mis Viajes”** en nuestro sitio web.

Recuerda que, si tu vuelo ha sido adelantado 16 minutos o más y no estás conforme con el nuevo horario, puedes cambiarlo sin costo (manteniendo el mismo destino y cabina) o solicitar la devolución del pasaje, siempre dentro de los 12 meses post

## 11. Notas para el informe técnico

- Guarda capturas de las celdas de la demo (sección 10) para el informe y la presentación (evidencia IE6/IE9), incluyendo el caso sin código de reserva (pregunta 3), que muestra cómo el agente pide más datos en vez de asumirlos.
- `reservas_latam.csv` es el dataset simulado; adjúntalo en el repositorio como evidencia.
- Este notebook usa el mismo stack que `RA1/IL1.1` (LangChain + Groq) y `RA1/IL1.3` (RAG vectorial con FAISS + Gemini embeddings) del curso — cítalo así en la sección de metodología/herramientas del informe.
- Explica en el informe por qué se separaron los índices FAISS (privacidad de la reserva) y por qué se armó el prompt manualmente en vez de usar `RetrievalQA` (que solo soporta una fuente).
- Las referencias APA de las 5 páginas oficiales de LATAM usadas como fuente van en la sección de Referencias del informe.